# dependencies

In [11]:
from pydantic import BaseModel, Field
from typing import List, Optional, Any, Dict
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from datetime import datetime
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import TavilySearchResults
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    ToolMessage
)
from langgraph.graph import StateGraph, END

import logging
import sys
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(filename='logs/reflexion_agent.log', mode='a')
    ]
)
logger = logging.getLogger(__name__)

from dotenv import load_dotenv
load_dotenv()
logger.info("✅ Environment variables loaded")

2025-08-24 17:16:26 | INFO     | __main__ | ✅ Environment variables loaded


In [12]:
MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.7
MAX_ITERATIONS = 2  # number of search + revision cycles

# schema definitions

In [13]:
class Reflection(BaseModel):
    missing: str = Field(description="Critique of what is missing.")
    superfluous: str = Field(description="Critique of what is superfluous.")

class AnswerQuestion(BaseModel):
    answer: str = Field(description="~250 word detailed answer to the question.")
    search_queries: List[str] = Field(description="1-3 search queries to improve the answer.")
    reflection: Reflection = Field(description="Self-critique of the initial answer.")

class ReviseAnswer(AnswerQuestion):
    references: List[str] = Field(description="Supporting citation URLs or identifiers.")

# prompt templates

In [14]:
BASE_SYSTEM_INSTRUCTIONS = """You are an expert AI researcher.
Current time: {time}

Process:
1. {first_instruction}
2. Reflect and critique your answer (be rigorous).
3. After the reflection, list 1-3 focused search queries (NOT inside the reflection text).
Return only the structured tool output.
"""

draft_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", BASE_SYSTEM_INSTRUCTIONS),
        MessagesPlaceholder(variable_name="messages"),
        ("system", "Return only a valid tool call matching the schema."),
    ]
).partial(time=lambda: datetime.now().isoformat())

REVISION_INSTRUCTIONS = """Revise your previous answer using new evidence:
- Incorporate improvements from critique + search results.
- Remove fluff / repetition.
- Keep within ~250 words.
- Add inline numeric citations [1], [2], etc.
- Append a References section (not in word count)."""

revision_prompt = draft_prompt.partial(first_instruction=REVISION_INSTRUCTIONS)
first_draft_prompt = draft_prompt.partial(first_instruction="Provide a detailed ~250 word answer.")


# llm init

In [15]:
def build_llm(model: str = MODEL, temperature: float = TEMPERATURE) -> ChatGoogleGenerativeAI:
    try:
        llm = ChatGoogleGenerativeAI(model=model, temperature=temperature)
        logger.info(f"✅ Loaded GoogleGemini model: {model} with temperature {temperature}")
        return llm
    except Exception as e:
        logger.error(f"❌ Error loading GoogleGemini model: {e}")
        raise

# tool-bound chains

In [16]:
def build_chains(llm: ChatGoogleGenerativeAI):
    try:
        draft_chain = first_draft_prompt | llm.bind_tools(
            tools=[AnswerQuestion],
            tool_choice="AnswerQuestion",
        )

        revision_chain = revision_prompt | llm.bind_tools(
            tools=[ReviseAnswer],
            tool_choice="ReviseAnswer",
        )

        logger.info("✅ Successfully built draft and revision chains")
        return draft_chain, revision_chain
    except Exception as e:
        logger.error(f"❌ Error building chains: {e}")
        raise

# search executor

In [17]:
class SearchExecutor:
    def __init__(self, max_results: int = 5):
        self.tool = TavilySearchResults(max_results=max_results, search_depth='basic')

    def run(self, state: List[BaseMessage]) -> List[ToolMessage]:
        if not state:
            logger.warning("⚠️ No state provided to SearchExecutor")
            return []
        last = state[-1]
        if not isinstance(last, AIMessage):
            logger.warning("⚠️ Last message is not from AI; skipping search execution")
            return []
        
        tool_calls = getattr(last, "tool_calls", []) or []
        if not tool_calls:
            logger.debug("ℹ️ No tool calls found in the last AI message")
            return []
        
        tool_messages = []
        for call in tool_calls:
            name = call.get("name")
            if name not in ["AnswerQuestion", "ReviseAnswer"]:
                logger.debug(f"ℹ️ Skipping unsupported tool call: {name}")
                continue

            call_id = call.get("id")
            args = call.get("args", {})
            queries = args.get("search_queries", [])
            logger.info(f"🔍 Executing {len(queries)} search queries for call_id: {call_id}")

            aggregated: Dict[str, Any] = {}
            for q in queries:
                try:
                    logger.info(f"🔎 Searching for query: {q}")
                    result = self.tool.invoke(q)
                    aggregated[q] = result
                    logger.debug(f"✅ Search results for '{q}': {str(result)[:250]}...")
                except Exception as e:
                    logger.error(f"❌ Error executing search query '{q}': {e}")
                    continue
            
            tool_messages.append(
                ToolMessage(
                    content=aggregated,
                    tool_call_id=call_id,
                )
            )
            logger.info(f"✅ Completed search for call_id: {call_id}")

        return tool_messages

# reflexion agent

In [18]:
class ReflexionAgent:
    def __init__(
            self,
            model: str = MODEL,
            temperature: float = TEMPERATURE,
            max_iterations: int = MAX_ITERATIONS,
    ):
        self.model = model
        self.temperature = temperature
        self.max_iterations = max_iterations
        self.llm = build_llm(model=self.model, temperature=self.temperature)
        self.draft_chain, self.revision_chain = build_chains(llm=self.llm)
        self.search_executor = SearchExecutor()
        self._build_graph()
        logger.info("✅ ReflexionAgent initialized successfully")

    # graph
    def _build_graph(self):
        graph = StateGraph(list)

        graph.add_node('draft', self._draft_node, name="Draft Answer")
        graph.add_node('execute_tools', self._execute_tools_node, name="Execute Tools")
        graph.add_node('revise', self._revise_node, name="Revise Answer")

        graph.set_entry_point('draft')
        graph.add_edge('draft', 'execute_tools')
        graph.add_edge('execute_tools', 'revise')
        graph.add_conditional_edges('revise', self._should_continue)

        self.app = graph.compile()
        logger.info("✅ StateGraph compiled successfully")

    # nodes
    def _draft_node(self, state: List[BaseMessage]) -> List[BaseMessage]:
        ai: AIMessage = self.draft_chain.invoke({"messages": state})
        if ai.content:
            logger.debug(f"AI Draft Response: {ai.content}")
        
        return state + [ai]
    
    def _execute_tools_node(self, state: List[BaseMessage]) -> List[BaseMessage]:
        tool_msgs = self.search_executor.run(state)
        logger.debug(f"Tool Messages: {len(tool_msgs)}")

        return state + tool_msgs
    
    def _revise_node(self, state: List[BaseMessage]) -> List[BaseMessage]:
        ai: AIMessage = self.revision_chain.invoke({"messages": state})
        if ai.content:
            logger.debug(f"AI Revision Response: {ai.content}")
        
        return state + [ai]
    
    # iteration control
    def _should_continue(self, state: List[BaseMessage]) -> str:
        tool_msgs = sum(isinstance(m, ToolMessage) for m in state)
        logger.info(f"Iteration controller: tool_messages={tool_msgs}/{self.max_iterations}")
        if tool_msgs >= self.max_iterations:
            logger.info("🔚 Reached max iterations; ending process")
            return END
        
        return 'execute_tools'
    
    # public api
    def run(self, question: str) -> Dict[str, Any]:
        logger.info(f"🚀 Starting ReflexionAgent with question: {question}")
        state: List[BaseMessage] = self.app.invoke([HumanMessage(content=question)])

        drafts: List[Dict[str, Any]] = []
        revisions: List[Dict[str, Any]] = []
        final_revision: Optional[Dict[str, Any]] = None

        for msg in state:
            if isinstance(msg, AIMessage) and msg.tool_calls:
                for call in msg.tool_calls:
                    name = call.get("name")
                    args = call.get("args", {})
                    if name == "AnswerQuestion":
                        drafts.append(args)
                    elif name == "ReviseAnswer":
                        revisions.append(args)
                        final_revision = args
        
        summary = {
            "question": question,
            "drafts": drafts,
            "revisions": revisions,
            "final_answer": (final_revision or {}).get("answer"),
            "final_references": (final_revision or {}).get("references"),
            "iterations": len(revisions),
            "raw_messages": state,
        }

        logger.info("✅ ReflexionAgent run completed")
        logger.info(f"Final Summary: \n{summary}")

        return summary
    
    def visualize_graph(self):
        print('=== Mermaid Graph ===')
        print(self.app.get_graph().draw_mermaid())
        print('=== ASCII Graph ===')
        print(self.app.get_graph().draw_ascii())

# main function

In [ ]:
def main():
    agent = ReflexionAgent()
    question = "What are the latest advancements in AI agents?"
    result = agent.run(question)
    print("\n=== FINAL ANSWER ===\n")
    print(result["final_answer"])

In [20]:
main()

2025-08-24 17:16:26 | INFO     | __main__ | ✅ Loaded GoogleGemini model: gemini-2.5-flash with temperature 0.7
2025-08-24 17:16:26 | INFO     | __main__ | ✅ Successfully built draft and revision chains
2025-08-24 17:16:26 | INFO     | __main__ | ✅ StateGraph compiled successfully
2025-08-24 17:16:26 | INFO     | __main__ | ✅ ReflexionAgent initialized successfully
2025-08-24 17:16:26 | INFO     | __main__ | 🚀 Starting ReflexionAgent with question: What are the latest advancements in AI agents?
2025-08-24 17:16:35 | INFO     | __main__ | 🔍 Executing 3 search queries for call_id: 4a95574b-0982-49a2-b1a7-93b6be345c93
2025-08-24 17:16:35 | INFO     | __main__ | 🔎 Searching for query: latest AI agent architectures and projects 2024-2025
2025-08-24 17:16:35 | DEBUG    | urllib3.connectionpool | Starting new HTTPS connection (1): api.tavily.com:443
2025-08-24 17:16:37 | DEBUG    | urllib3.connectionpool | https://api.tavily.com:443 "POST /search HTTP/1.1" 200 1980
2025-08-24 17:16:37 | DEBUG 